# 14 | LangChain LCEL：从模型输出到业务动作

很多刚接触LangChain/LLM的人都会好奇一个点，不是模型不会回答，而是模型回答完以后，你的业务系统不知道下一步该干什么，怎么干。

比如用户说：

```text
我昨天买的蓝牙耳机还没发货，不想要了，帮我退一下。
```

一个真正可用的客服系统，不能只回复“好的，我帮你处理”。它还要完成：

```text
识别退款意图 -> 提取工单字段 -> 标准化字段 -> 判断处理通道 -> 执行业务动作 -> 生成用户回复
```

![LCEL 从模型输出到业务动作](assets/14-lcel-business-flow.svg)

本文就用这个退款工单场景，讲清楚三个组件如何把“模型输出”变成“业务动作”：

- `JsonOutputParser`：把模型输出变成结构化工单
- `RunnableLambda`：把结构化工单交给确定性业务规则处理
- `StrOutputParser`：把最终模型回复变成可展示文本

一条 LangChain 链，本质上是一条数据流水线：

```text
输入 -> Prompt -> Model -> Parser -> RunnableLambda -> Prompt -> Model -> 输出
```

这些组件不负责让模型更聪明，而是负责让链里每一步的输入输出接得上。链路接不上，再聪明的模型也只能在旁边干着急。

## 一、核心组件速览

| 组件 | 输入 | 输出 | 典型用途 |
| --- | --- | --- | --- |
| `StrOutputParser` | `AIMessage` / 模型输出 | `str` | 最终结果给人看 |
| `JsonOutputParser` | `AIMessage` / 模型输出 | `dict` / JSON 对象 | 结果继续传给链后续组件 |
| `RunnableLambda` | 任意对象 | 任意对象 | 在链中插入确定性 Python 逻辑 |

**Parser 负责改数据形状，RunnableLambda 负责插入确定性规则。**

## 二、示例场景：智能客服退款工单

假设用户输入：

```text
我昨天买的蓝牙耳机还没发货，不想要了，帮我退一下。
```

系统要做四件事：

1. 从自然语言里提取结构化信息
2. 把模型提取的字段转成系统标准值
3. 根据处理通道生成下一步动作
4. 生成一句可以给用户看的客服回复

这四个步骤正好对应：

```text
JsonOutputParser -> RunnableLambda -> RunnableLambda -> StrOutputParser
```

## 三、初始化模型与解析器

这里通过阿里云百炼的 OpenAI 兼容接口调用聊天模型。

项目根目录 `.env` 中需要配置 `DASHSCOPE_API_KEY` 和 `DASHSCOPE_BASE_URL`。模型名称可以按当前账号可用模型调整。

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
import os

# 聊天模型负责理解用户输入和生成回复。
llm = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

# JsonOutputParser：把模型输出解析成 Python dict。
json_parser = JsonOutputParser()

# StrOutputParser：把模型输出解析成普通字符串。
str_parser = StrOutputParser()

## 四、JsonOutputParser：提取结构化工单

第一步，让模型从用户话里提取退款工单信息。

这里的输出不是最终给用户看的，而是要继续传给后面的链组件。

In [ ]:
extract_prompt = PromptTemplate.from_template(
    """
你是电商客服系统的信息抽取器。

请从用户消息中提取退款工单字段，只输出 JSON，不要输出解释。

字段要求：
- intent: 用户意图，例如 refund、exchange、consult
- product: 商品名称
- reason: 退款原因
- shipped: 是否已发货，true / false / null
- urgency: 紧急程度，low / medium / high

用户消息：{message}
    """.strip()
)

# 链路：用户消息 -> Prompt -> 模型 -> JSON 解析成 dict
extract_chain = extract_prompt | llm | json_parser

ticket = extract_chain.invoke(
    {"message": "我昨天买的蓝牙耳机还没发货，不想要了，帮我退一下。"}
)

ticket

这一步之后，链里的数据已经不是一段文字，而是一个 `dict`。

它可以继续传给函数、PromptTemplate、工具、函数节点或下一个模型。重点不是“给程序用”还是“给模型用”，而是：**给链的后续组件用。**

## 五、RunnableLambda：注入确定性业务规则

模型输出的字段未必完全符合系统标准。比如：

- `intent` 可能是 `退款`，但系统只认 `refund`
- `urgency` 可能缺失，需要补默认值
- 未发货退款可以自动进入快速退款通道

这类确定性逻辑不要交给模型猜，应该用 Python 函数处理。

In [ ]:
def normalize_ticket(ticket: dict) -> dict:
    """把模型提取结果转换成系统更稳定可用的工单对象。"""
    intent_map = {
        "退款": "refund",
        "退货": "refund",
        "refund": "refund",
        "换货": "exchange",
        "exchange": "exchange",
        "咨询": "consult",
        "consult": "consult",
    }

    normalized = dict(ticket)
    normalized["intent"] = intent_map.get(ticket.get("intent"), ticket.get("intent", "consult"))
    normalized["urgency"] = ticket.get("urgency") or "medium"

    # 未发货退款可以走快速退款通道。
    # route 不是展示用的漂亮话，而是后续流程要读取的机器字段。
    normalized["route"] = "fast_refund" if normalized.get("intent") == "refund" and normalized.get("shipped") is False else "manual_review"

    return normalized


# 标准写法：用 RunnableLambda 把 Python 函数包装成可入链组件。
normalize_runnable = RunnableLambda(normalize_ticket)

In [ ]:
# 链路：提取结构化工单 -> 标准化字段 -> 判断 route
extract_and_normalize_chain = extract_prompt | llm | json_parser | normalize_runnable

normalized_ticket = extract_and_normalize_chain.invoke(
    {"message": "我昨天买的蓝牙耳机还没发货，不想要了，帮我退一下。"}
)

normalized_ticket

`fast_refund` 的意思不是“已经退款成功”，而是：这张工单会进入后续的快速退款流程。

比如真实系统里，后面可能还会做这些动作：

- 检查订单是否未发货
- 锁定发货流程
- 创建退款申请
- 通知支付系统处理退款
- 给用户发送进度提醒

`RunnableLambda` 的价值是：把模型不擅长的稳定规则放回代码里。

模型负责理解语言，函数负责执行规则。分工明确，少掉很多玄学。

## 六、基于 route 生成并执行业务动作

既然 `route` 是机器字段，就应该继续驱动后面的流程。

下面先用一个 `RunnableLambda` 把 `route` 转成动作清单，再用另一个 `RunnableLambda` 模拟执行业务动作。

真实项目中，执行器会调用订单、售后、支付或消息系统。


In [ ]:
def build_next_actions(ticket: dict) -> dict:
    """根据处理通道生成后续动作。"""
    actions_by_route = {
        "fast_refund": [
            "校验订单未发货",
            "锁定发货流程",
            "创建退款申请",
            "通知用户退款请求已进入优先处理",
        ],
        "manual_review": [
            "转人工客服复核",
            "补充订单状态和售后原因",
        ],
    }

    enriched = dict(ticket)
    enriched["next_actions"] = actions_by_route.get(ticket.get("route"), actions_by_route["manual_review"])
    return enriched


next_actions_runnable = RunnableLambda(build_next_actions)

# 链路：标准化工单 -> 根据 route 生成后续动作清单
workflow_chain = extract_prompt | llm | json_parser | normalize_runnable | next_actions_runnable

workflow_ticket = workflow_chain.invoke(
    {"message": "我昨天买的蓝牙耳机还没发货，不想要了，帮我退一下。"}
)

workflow_ticket

动作清单生成后，真实系统通常会进入一个执行器：

```text
route = fast_refund
-> 校验订单状态
-> 锁定发货流程
-> 创建退款申请
-> 通知用户
```

下面用一个模拟执行器表示这件事。真实项目里，这里会替换成订单服务、售后系统、消息系统的 API 调用。


In [ ]:
def execute_actions(ticket: dict) -> dict:
    """模拟执行业务动作。真实系统里会在这里调用订单、退款、通知等服务。"""
    executed = []

    for action in ticket.get("next_actions", []):
        # Demo示例只记录动作。
        # 真实项目中可以替换成 order_api.check_status(...)、refund_api.create(...) 等。
        executed.append(f"已执行：{action}")

    result = dict(ticket)
    result["executed_actions"] = executed
    return result


execute_actions_runnable = RunnableLambda(execute_actions)

# 链路：生成动作清单 -> 模拟执行业务动作
workflow_with_execution_chain = (
    extract_prompt
    | llm
    | json_parser
    | normalize_runnable
    | next_actions_runnable
    | execute_actions_runnable
)

executed_ticket = workflow_with_execution_chain.invoke(
    {"message": "我昨天买的蓝牙耳机还没发货，不想要了，帮我退一下。"}
)

executed_ticket


## 七、StrOutputParser：生成面向用户的回复

最后一步，根据标准化后的工单，生成客服回复。

这次结果是给用户看的，所以用 `StrOutputParser` 转成普通字符串。

In [ ]:
reply_prompt = PromptTemplate.from_template(
    """
你是电商客服助手。请根据工单信息生成一句简洁、礼貌的回复。

工单信息：
- 意图：{intent}
- 商品：{product}
- 原因：{reason}
- 是否已发货：{shipped}
- 处理通道：{route}
- 已执行动作：{executed_actions}

要求：
- 不要编造订单号
- 不要承诺已经退款成功
- 如果是 fast_refund，说明退款请求已进入优先处理流程
    """.strip()
)

# 链路：完成工单处理 -> 生成面向用户的最终回复 -> 转成字符串
reply_chain = extract_prompt | llm | json_parser | normalize_runnable | next_actions_runnable | execute_actions_runnable | reply_prompt | llm | str_parser

reply = reply_chain.invoke(
    {"message": "我昨天买的蓝牙耳机还没发货，不想要了，帮我退一下。"}
)

print(reply)

这条链里的每个节点都对应一次数据转换：

```text
输入 dict                    # {"message": "帮我退款"}
-> extract_prompt            # dict -> PromptValue，生成信息抽取提示词
-> llm                       # PromptValue -> AIMessage，生成 JSON 文本
-> json_parser               # AIMessage -> dict，解析成结构化工单
-> normalize_runnable        # dict -> dict，统一字段并判断 route
-> next_actions_runnable     # dict -> dict，根据 route 生成动作清单
-> execute_actions_runnable  # dict -> dict，模拟执行业务动作
-> reply_prompt              # dict -> PromptValue，生成客服回复提示词
-> llm                       # PromptValue -> AIMessage，生成客服回复
-> str_parser                # AIMessage -> str，转成最终可展示文本
```

这正好对应完整链路：

```python
reply_chain = extract_prompt | llm | json_parser | normalize_runnable | next_actions_runnable | execute_actions_runnable | reply_prompt | llm | str_parser
```

这就是 Parser 和 RunnableLambda 的核心意义：让链路里的数据类型始终对得上。

## 八、选型建议

### 1. `StrOutputParser`：面向展示的文本输出

当模型输出就是最终答案，直接给用户看。

典型例子：

- 客服回复
- 文章总结
- 报告草稿
- 问答结果

### 2. `JsonOutputParser`：面向链路传递的结构化输出

当模型输出还要继续传给链后面的组件。

典型例子：

- 从用户话里提取订单字段
- 判断用户意图
- 生成工具调用参数
- 把文本分类成结构化标签

### 3. `RunnableLambda`：面向规则处理的函数节点

当链中需要插入确定性业务逻辑。

典型例子：

- 字段标准化
- 参数校验
- 补默认值
- 路由判断
- 格式转换

一句话：**能写死的规则，就别让模型即兴发挥。模型不是不能算，但它有时候像一个临场发挥型同事。**

## 九、总结

记住这三句话：

- `StrOutputParser`：把模型输出变成最终可展示的字符串
- `JsonOutputParser`：把模型输出变成链后续组件可使用的结构化对象
- `RunnableLambda`：把 Python 函数变成 LCEL 链里的一个步骤

最常见的组合是：

```python
chain = first_prompt | llm | JsonOutputParser() | RunnableLambda(clean_data) | RunnableLambda(build_actions) | RunnableLambda(execute_actions) | second_prompt | llm | StrOutputParser()
```

前半段负责理解、结构化和执行业务动作，后半段负责生成给人看的答案。

参考资料：

- LangChain Output Parsers：https://api.python.langchain.com/en/latest/core/output_parsers.html
- LangChain JsonOutputParser：https://api.python.langchain.com/en/latest/output_parsers/langchain_core.output_parsers.json.JsonOutputParser.html
- LangChain RunnableLambda：https://api.python.langchain.com/en/latest/core/runnables/langchain_core.runnables.base.RunnableLambda.html